# Xem nhanh cấu trúc sample data MBD-mini

Notebook hiển thị **5 dòng đầu của từng bảng** trong `sampledata/mbd_mini/`, kèm tên cột và kiểu dữ liệu Parquet/Arrow.
Sample gồm 1.000 khách được lấy mẫu phân tầng; xem `sampledata/README.md` để biết phương pháp và giới hạn của mẫu.

**Cách chạy:** chọn kernel Python trong `.venv`, sau đó **Run All**.
Nếu thiếu thư viện, chạy trong terminal ở thư mục gốc repo:

```powershell
.\.venv\Scripts\python.exe -m pip install pandas pyarrow ipykernel
```

Mỗi bảng được đọc trực tiếp từ file Parquet sample, lấy theo thứ tự dòng trong file, không sắp xếp theo thời gian.
Cột `fold` chỉ hiển thị nếu có sẵn trong bảng, không suy ra từ thư mục.
Các mảng dài được rút gọn khi hiển thị; DataFrame mẫu vẫn giữ nguyên giá trị.


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import Markdown, display

# Tìm repo khi chạy từ thư mục gốc, notebooks/ hoặc thư mục con.
PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "sampledata" / "mbd_mini").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm thấy sampledata/mbd_mini/. Hãy chạy notebook trong repo này.")

DATA_DIR = PROJECT_ROOT / "sampledata" / "mbd_mini"
N_ROWS = 5
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_seq_items", 6)
np.set_printoptions(threshold=6, edgeitems=2)
print(f"Thư mục dữ liệu: {DATA_DIR}")


Thư mục dữ liệu: c:\Users\admin\Documents\Thực tập VSF\new propensity\sampledata\mbd_mini


## Các bảng dữ liệu

`dialog`, `geo`, `trx` lưu từng sự kiện; `ptls_*` gom chuỗi sự kiện của khách hàng thành các mảng.
`clients` chứa khách hàng và fold, còn `targets` chứa nhãn theo tháng.
Các bảng phụ mô tả việc lấy mẫu và lưới khách hàng theo tháng.


In [4]:
TABLES = {
    "clients": "Khách hàng được chọn và fold",
    "targets": "Nhãn sản phẩm và thông tin theo tháng",
    "dialog": "Từng lượt hội thoại và embedding",
    "geo": "Từng sự kiện vị trí",
    "trx": "Từng giao dịch",
    "ptls_dialog": "Chuỗi hội thoại của khách hàng",
    "ptls_geo": "Chuỗi sự kiện vị trí của khách hàng",
    "ptls_trx": "Chuỗi giao dịch của khách hàng",
    "selection": "Tầng lấy mẫu và trọng số của khách hàng",
    "customer_month": "Lưới khách hàng theo tháng và số log",
    "coverage_selection": "Khách minh họa theo nhóm có chủ đích",
}
parquet_files = {name: DATA_DIR / f"{name}.parquet" for name in TABLES}
missing = [str(path) for path in parquet_files.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Không tìm thấy file Parquet: {missing}")

display(pd.DataFrame([
    {"Bảng": name, "Mô tả": description, "File Parquet": parquet_files[name].name}
    for name, description in TABLES.items()
]))


,Bảng,Mô tả,File Parquet
0,clients,Khách hàng được chọn và fold,clients.parquet
1,targets,Nhãn sản phẩm và thông tin theo tháng,targets.parquet
2,dialog,Từng lượt hội thoại và embedding,dialog.parquet
3,geo,Từng sự kiện vị trí,geo.parquet
4,trx,Từng giao dịch,trx.parquet
5,ptls_dialog,Chuỗi hội thoại của khách hàng,ptls_dialog.parquet
6,ptls_geo,Chuỗi sự kiện vị trí của khách hàng,ptls_geo.parquet
7,ptls_trx,Chuỗi giao dịch của khách hàng,ptls_trx.parquet
8,selection,Tầng lấy mẫu và trọng số của khách hàng,selection.parquet
9,customer_month,Lưới khách hàng theo tháng và số log,customer_month.parquet


## Hàm xem mẫu

Đọc batch đầu tiên từ mỗi file Parquet sample, tối đa 5 dòng.
Bảng schema hiển thị kiểu dữ liệu gốc, bao gồm thời gian và mảng lồng nhau.


In [5]:
def preview_table(name):
    path = parquet_files[name]
    with pq.ParquetFile(path) as parquet:
        batch = next(parquet.iter_batches(batch_size=N_ROWS), None)
        sample = (
            pa.Table.from_batches([batch])
            if batch is not None
            else pa.Table.from_batches([], schema=parquet.schema_arrow)
        )
    frame = sample.to_pandas()

    display(Markdown(f"**{len(frame)} dòng mẫu · {len(sample.schema)} cột**"))
    display(frame)
    display(Markdown("**Tên cột và kiểu dữ liệu**"))
    display(pd.DataFrame([
        {"Cột": field.name, "Kiểu Arrow": str(field.type), "Kiểu pandas": str(frame[field.name].dtype)}
        for field in sample.schema
    ]))
    return frame


## clients

Khách hàng và fold dùng cho cross-validation.


In [6]:
clients = preview_table("clients")


**5 dòng mẫu · 2 cột**

,client_id,fold
0,7adb539fba5bdf3f1528f33050c70939a038496ad554a6d7a51f3479ca16a36a,0
1,b146d5c6c1bdaacf88784dac3d0862b16291432a75ce460722591728d98fb539,0
2,c7417a580f536366dffc90527a627bfc7aa6c8b49f48e12455c8cea7cbc4dee9,0
3,b9a7abf182facadb8e33e906fb40ed53c59bc4170f8ade18882eb4a420e46e4d,0
4,40cebd78f5065d35d3c730d8e98a4009bb5b9675ef14f5db3c61a9833079c2d7,0


**Tên cột và kiểu dữ liệu**

,Cột,Kiểu Arrow,Kiểu pandas
0,client_id,string,str
1,fold,int64,int64


## targets

Nhãn sản phẩm và thông tin khách hàng theo tháng.


In [7]:
targets = preview_table("targets")


**5 dòng mẫu · 9 cột**

,client_id,mon,target_1,target_2,target_3,target_4,trans_count,diff_trans_date,fold
0,048a83de1385297487292d860ea9bd16538c85ad93b72ac3d14c3f2f79025af8,2022-02-28,0,0,0,0,0,NaN,0
1,048a83de1385297487292d860ea9bd16538c85ad93b72ac3d14c3f2f79025af8,2022-03-31,0,0,0,0,3,6.0,0
2,048a83de1385297487292d860ea9bd16538c85ad93b72ac3d14c3f2f79025af8,2022-04-30,0,0,0,0,7,2.0,0
3,048a83de1385297487292d860ea9bd16538c85ad93b72ac3d14c3f2f79025af8,2022-05-31,0,0,0,0,9,4.0,0
4,048a83de1385297487292d860ea9bd16538c85ad93b72ac3d14c3f2f79025af8,2022-06-30,0,0,0,0,12,0.0,0


**Tên cột và kiểu dữ liệu**

,Cột,Kiểu Arrow,Kiểu pandas
0,client_id,string,str
1,mon,string,str
2,target_1,int32,int32
3,target_2,int32,int32
4,target_3,int32,int32
5,target_4,int32,int32
6,trans_count,int32,int32
7,diff_trans_date,int32,float64
8,fold,int64,int64


## dialog

Mỗi dòng là một sự kiện hội thoại, có embedding dạng mảng.


In [8]:
detail_dialog = preview_table("dialog")


**5 dòng mẫu · 4 cột**

,client_id,event_time,embedding,fold
0,4e70f83ad96d8ce59b2fa7c7121e0a23a3add1f65a3d549eef22bb87f924c2c5,2022-12-03 05:35:02.876821,"[0.27138, -0.179337, 0.467741, -0.277448, -0.032484, 0.451949, ...]",0
1,fd0500c0fd819f25d4082a549509ee5b805b5b444974d2d7b854bc64a47c1866,2022-04-04 10:51:22.591313,"[-0.081687, 0.041271, 0.280346, 0.205108, 0.159652, 0.280946, ...]",0
2,fd0500c0fd819f25d4082a549509ee5b805b5b444974d2d7b854bc64a47c1866,2022-08-11 07:59:39.535432,"[0.237039, -0.363748, 0.477344, -0.286838, -0.219207, 0.42359, ...]",0
3,fd0500c0fd819f25d4082a549509ee5b805b5b444974d2d7b854bc64a47c1866,2022-03-05 10:02:30.330544,"[0.324057, -0.170505, 0.503197, -0.411453, -0.239541, 0.495105, ...]",0
4,fd0500c0fd819f25d4082a549509ee5b805b5b444974d2d7b854bc64a47c1866,2022-12-13 10:27:32.114652,"[0.16858, 0.000878, 0.335482, 0.015159, -0.083275, 0.317838, ...]",0


**Tên cột và kiểu dữ liệu**

,Cột,Kiểu Arrow,Kiểu pandas
0,client_id,string,str
1,event_time,timestamp[us],datetime64[us]
2,embedding,list<element: float>,object
3,fold,int64,int64


## geo

Mỗi dòng là một sự kiện vị trí.


In [9]:
detail_geo = preview_table("geo")


**5 dòng mẫu · 6 cột**

,client_id,event_time,geohash_4,geohash_5,geohash_6,fold
0,c89e9d1fd63c91945b92e7973ed35c21a7ceb789bc17a4eafd88ea778df7593c,2022-12-04 13:04:35.613738,48959,163587,1898292,0
1,c89e9d1fd63c91945b92e7973ed35c21a7ceb789bc17a4eafd88ea778df7593c,2022-01-22 08:04:12.292772,48959,311429,831581,0
2,c89e9d1fd63c91945b92e7973ed35c21a7ceb789bc17a4eafd88ea778df7593c,2022-11-30 14:45:27.047474,30094,95818,2362661,0
3,c89e9d1fd63c91945b92e7973ed35c21a7ceb789bc17a4eafd88ea778df7593c,2022-11-13 13:22:26.851583,30094,95818,2362661,0
4,c89e9d1fd63c91945b92e7973ed35c21a7ceb789bc17a4eafd88ea778df7593c,2022-11-11 11:30:10.183928,30094,95818,2362661,0


**Tên cột và kiểu dữ liệu**

,Cột,Kiểu Arrow,Kiểu pandas
0,client_id,string,str
1,event_time,timestamp[us],datetime64[us]
2,geohash_4,int32,int32
3,geohash_5,int32,int32
4,geohash_6,int32,int32
5,fold,int64,int64


## trx

Mỗi dòng là một giao dịch.


In [10]:
detail_trx = preview_table("trx")


**5 dòng mẫu · 15 cột**

,client_id,event_time,amount,event_type,event_subtype,currency,src_type11,src_type12,dst_type11,dst_type12,src_type21,src_type22,src_type31,src_type32,fold
0,c6623541f11e74d361e2337fd7be4033f2531423880af8a1b1bf3c658e52e84c,2021-12-18 19:40:40.609271,3960.616943,39,26,11,105,709,168,2850,3809,8,453,69,0
1,c6623541f11e74d361e2337fd7be4033f2531423880af8a1b1bf3c658e52e84c,2021-05-08 22:59:26.600901,1187.765381,39,26,11,105,709,168,2850,3809,8,453,69,0
2,c6623541f11e74d361e2337fd7be4033f2531423880af8a1b1bf3c658e52e84c,2021-08-30 19:21:04.625151,4404.545410,39,26,11,105,709,168,2850,3809,8,453,69,0
3,c6623541f11e74d361e2337fd7be4033f2531423880af8a1b1bf3c658e52e84c,2021-11-06 19:25:10.285439,5549.162109,39,26,11,105,709,168,2850,3809,8,453,69,0
4,c6623541f11e74d361e2337fd7be4033f2531423880af8a1b1bf3c658e52e84c,2021-09-18 20:25:13.055857,3443.130615,39,26,11,105,709,168,2850,3809,8,453,69,0


**Tên cột và kiểu dữ liệu**

,Cột,Kiểu Arrow,Kiểu pandas
0,client_id,string,str
1,event_time,timestamp[us],datetime64[us]
2,amount,float,float32
3,event_type,int32,int32
4,event_subtype,int32,int32
5,currency,int32,int32
6,src_type11,int32,int32
7,src_type12,int32,int32
8,dst_type11,int32,int32
9,dst_type12,int32,int32


## ptls_dialog

Dữ liệu hội thoại dạng chuỗi; một số cột là mảng hoặc mảng lồng nhau.


In [11]:
ptls_dialog = preview_table("ptls_dialog")


**5 dòng mẫu · 4 cột**

,client_id,event_time,embedding,fold
0,b70515f4781266435175efb09cbfc0cb37888e4c74e9bc267c67966cf25d5138,"[1656844090, 1658487124]","[[0.299247, -0.150526, 0.458324, -0.293501, -0.259607, 0.455546, ...], [0.390311, -0.029613, 0.5...",0
1,87b105c6acc617b2bfe881c4fa5a66ab2873a8ff5affead0eef68cd1b7ac95e8,[1630904081],"[[0.397082, -0.364542, 0.569581, -0.574909, -0.379323, 0.45452, ...]]",0
2,5cf24741459ab1fd78c34c43f963f7b292ea7dcc660d1bce753dc0f36ac14565,[1665372370],"[[0.326814, -0.077865, 0.537796, -0.242592, -0.32559, 0.49748, ...]]",0
3,048a83de1385297487292d860ea9bd16538c85ad93b72ac3d14c3f2f79025af8,"[1637052186, 1637584951]","[[0.035546, -0.029841, 0.35443, 0.091589, -0.155762, 0.329505, ...], [0.385071, -0.205584, 0.488...",0
4,8c57f0b6b9642860e2d3405b566f2990b86b54ed4615881d1843a232f7d57289,"[1622955131, 1623761514, 1624102070, 1636951929, 1638776174]","[[0.326388, -0.218458, 0.455793, -0.512267, -0.261366, 0.540589, ...], [0.55935, -0.371663, 0.51...",0


**Tên cột và kiểu dữ liệu**

,Cột,Kiểu Arrow,Kiểu pandas
0,client_id,string,str
1,event_time,list<element: int64>,object
2,embedding,list<element: list<element: float>>,object
3,fold,int64,int64


## ptls_geo

Dữ liệu vị trí dạng chuỗi.


In [12]:
ptls_geo = preview_table("ptls_geo")


**5 dòng mẫu · 6 cột**

,client_id,event_time,geohash_4,geohash_5,geohash_6,fold
0,c89e9d1fd63c91945b92e7973ed35c21a7ceb789bc17a4eafd88ea778df7593c,"[1639825787, 1639863850, 1639921265, 1640264674, 1640264681, 1640358956, ...]","[404, 404, 559, 1179, 404, 404, ...]","[5792, 5792, 10001, 4494, 5792, 5792, ...]","[10001, 10001, 10001, 10001, 10001, 10001, ...]",0
1,5cf24741459ab1fd78c34c43f963f7b292ea7dcc660d1bce753dc0f36ac14565,"[1640918207, 1641035894, 1641204501, 1641204583, 1641237017, 1641407429, ...]","[199, 7, 7, 7, 199, 199, ...]","[709, 174, 174, 174, 1600, 1600, ...]","[8022, 5412, 5412, 5412, 10001, 10001, ...]",0
2,820e75f51e9735e96d6005cb36f457d9f8bfb4fb56509aa999ca6d62a19ed966,"[1639396878, 1639503598, 1639741706, 1639770217, 1639824065, 1640029481, ...]","[234, 234, 234, 234, 214, 234, ...]","[2009, 2009, 2009, 2009, 10001, 2009, ...]","[10001, 10001, 10001, 10001, 10001, 10001, ...]",0
3,e64ded22f7483ab0bdd85029b1828a7ac33e45b31e874cc1e7c195b5d6ba6820,"[1651059567, 1653931107]","[4716, 4716]","[10001, 10001]","[10001, 10001]",0
4,cc7480a81b7518adb347295d2a3d44cebbb05b452a2f60e15c173b1562b59385,"[1640511016, 1641027876, 1641643266, 1641738495, 1641809392, 1641810437, ...]","[16, 16, 16, 16, 16, 16, ...]","[496, 496, 107, 496, 496, 87, ...]","[10001, 10001, 2188, 4772, 7321, 1267, ...]",0


**Tên cột và kiểu dữ liệu**

,Cột,Kiểu Arrow,Kiểu pandas
0,client_id,string,str
1,event_time,list<element: int64>,object
2,geohash_4,list<element: int32>,object
3,geohash_5,list<element: int32>,object
4,geohash_6,list<element: int32>,object
5,fold,int64,int64


## ptls_trx

Dữ liệu giao dịch dạng chuỗi.


In [13]:
ptls_trx = preview_table("ptls_trx")


**5 dòng mẫu · 15 cột**

,client_id,event_time,amount,event_type,event_subtype,currency,src_type11,src_type12,dst_type11,dst_type12,src_type21,src_type22,src_type31,src_type32,fold
0,ef8a64b1e5d16529bb92e048e831e2021b4d12c96df2348f2339d892278e6fed,"[1610120539, 1610155667, 1610283391, 1610781973, 1610894099, 1610976812, ...]","[24549.40625, 2806.251221, 12562.759766, 274358.3125, 136209.71875, 83133.101562, ...]","[1, 1, 1, 1, 1, 1, ...]","[1, 1, 1, 1, 1, 1, ...]","[1, 1, 1, 1, 1, 1, ...]","[1, 1, 1, 1, 1, 1, ...]","[1, 1, 1, 1, 1, 1, ...]","[1, 1, 2, 2, 1, 1, ...]","[1, 1, 4, 4, 1, 1, ...]","[426, 426, 426, 426, 426, 426, ...]","[1, 1, 1, 1, 1, 1, ...]","[76, 76, 76, 76, 76, 76, ...]","[8, 8, 8, 8, 8, 8, ...]",0
1,048a83de1385297487292d860ea9bd16538c85ad93b72ac3d14c3f2f79025af8,"[1610323710, 1610338168, 1610503374, 1610739204, 1611159286, 1611459138, ...]","[104.681908, 949.446106, 9895.102539, 13686.396484, 83.508759, 9703.803711, ...]","[13, 13, 29, 12, 3, 29, ...]","[14, 14, 31, 10, 3, 31, ...]","[1, 1, 1, 1, 1, 1, ...]","[13, 13, 3, 3, 1, 3, ...]","[21, 21, 37, 38, 1, 37, ...]","[13, 13, 6, 3, 3, 6, ...]","[18, 18, 33, 3, 3, 33, ...]","[210, 210, 210, 210, 210, 210, ...]","[24, 24, 24, 24, 24, 24, ...]","[6, 6, 6, 6, 6, 6, ...]","[10, 10, 10, 10, 10, 10, ...]",0
2,c87c9229dd20e7001c7238bf41e9904bbb57c64a5aec96dfddcdf8b6677f163c,"[1610702207, 1611580121, 1613523807, 1613998630, 1615986330, 1616741661, ...]","[2462.559082, 55551.210938, 173948.5625, 9511.602539, 34396.847656, 90355.546875, ...]","[1, 1, 1, 1, 1, 1, ...]","[1, 1, 1, 1, 1, 1, ...]","[1, 1, 1, 1, 1, 1, ...]","[7, 7, 7, 7, 7, 7, ...]","[15, 15, 15, 15, 15, 15, ...]","[1, 1, 2, 1, 1, 2, ...]","[1, 1, 2, 1, 1, 2, ...]","[13, 13, 13, 13, 13, 13, ...]","[3, 3, 3, 3, 3, 3, ...]","[7, 7, 7, 7, 7, 7, ...]","[2, 2, 2, 2, 2, 2, ...]",0
3,605941bb082cf951a789a2060d4e86b90b6c42477b238887d11047b564e18628,"[1609469319, 1609511104, 1609545699, 1609566495, 1609605375, 1609613949, ...]","[4565.471191, 23102.152344, 91803.96875, 7606.307129, 10.921645, 4.791159, ...]","[12, 7, 1, 7, 3, 3, ...]","[10, 43, 1, 43, 3, 3, ...]","[1, 1, 1, 1, 1, 1, ...]","[3, 1, 1, 1, 1, 1, ...]","[11, 1, 1, 1, 1, 1, ...]","[3, 16, 1, 16, 3, 3, ...]","[3, 25, 1, 25, 3, 3, ...]","[1535, 1535, 1535, 1535, 1535, 1535, ...]","[39, 39, 39, 39, 39, 39, ...]","[77, 77, 77, 77, 77, 77, ...]","[3, 3, 3, 3, 3, 3, ...]",0
4,6690b079d6463de0d1a2020bfc1bb87af5bc8b9538f75071b9a837fa60e708d6,"[1611721905, 1612641353, 1612849250, 1612931217, 1613789279, 1613879694, ...]","[1284108.75, 892515.9375, 84764.09375, 39538.664062, 32.270905, 32345.179688, ...]","[2, 2, 2, 2, 1, 2, ...]","[2, 2, 2, 2, 1, 2, ...]","[1, 1, 1, 1, 1, 1, ...]","[2, 2, 2, 2, 2, 2, ...]","[8, 8, 8, 8, 8, 8, ...]","[6, 6, 2, 2, 1, 2, ...]","[7, 7, 2, 2, 1, 2, ...]","[4408, 4408, 4408, 4408, 4408, 4408, ...]","[78, 78, 78, 78, 78, 78, ...]","[206, 206, 206, 206, 206, 206, ...]","[79, 79, 79, 79, 79, 79, ...]",0


**Tên cột và kiểu dữ liệu**

,Cột,Kiểu Arrow,Kiểu pandas
0,client_id,string,str
1,event_time,list<element: int64>,object
2,amount,list<element: float>,object
3,event_type,list<element: int32>,object
4,event_subtype,list<element: int32>,object
5,currency,list<element: int32>,object
6,src_type11,list<element: int32>,object
7,src_type12,list<element: int32>,object
8,dst_type11,list<element: int32>,object
9,dst_type12,list<element: int32>,object


## selection


In [14]:
selection = preview_table("selection")


**5 dòng mẫu · 6 cột**

,client_id,fold,activity_band,source_mask,stratum,sampling_weight
0,006bfd08d10bc8d4be10adf89c9e82d30a0af6dea2060d7b81e78712f5974b26,3,4,7,3:7:4,99.727273
1,0103dcccc93df7925acf0436546e09f037a190b1cc05a2262b35ea9fb0cc8711,2,3,3,2:3:3,99.142857
2,017e798b45667bba27fd95095ae5a38dcccd03cd07eabca552101c22b137aa18,3,2,3,3:3:2,99.619048
3,0186ae45a4859d2021639c0d6f79028e6d9f063e17410bf19c0484a810a25e2f,4,2,5,4:5:2,93.714286
4,01bf36431f4b8c155b6b52e5c2c8756fc472d1a5ab3dd1b96941b4f5e602a5af,3,2,1,3:1:2,100.714286


**Tên cột và kiểu dữ liệu**

,Cột,Kiểu Arrow,Kiểu pandas
0,client_id,large_string,str
1,fold,int64,int64
2,activity_band,int64,int64
3,source_mask,uint8,uint8
4,stratum,large_string,str
5,sampling_weight,double,float64


## customer_month


In [15]:
customer_month = preview_table("customer_month")


**5 dòng mẫu · 11 cột**

,client_id,month,trx_events,trx_raw_events,trx_missing_reason,geo_events,geo_raw_events,geo_missing_reason,dialog_events,dialog_raw_events,dialog_missing_reason
0,006bfd08d10bc8d4be10adf89c9e82d30a0af6dea2060d7b81e78712f5974b26,2020-12,0,0,None,<NA>,0,unobserved_early_source,<NA>,0,unobserved_early_source
1,006bfd08d10bc8d4be10adf89c9e82d30a0af6dea2060d7b81e78712f5974b26,2021-01,16,16,None,<NA>,0,unobserved_early_source,<NA>,0,unobserved_early_source
2,006bfd08d10bc8d4be10adf89c9e82d30a0af6dea2060d7b81e78712f5974b26,2021-02,37,37,None,<NA>,0,unobserved_early_source,<NA>,0,unobserved_early_source
3,006bfd08d10bc8d4be10adf89c9e82d30a0af6dea2060d7b81e78712f5974b26,2021-03,33,33,None,<NA>,0,unobserved_early_source,1,1,NaN
4,006bfd08d10bc8d4be10adf89c9e82d30a0af6dea2060d7b81e78712f5974b26,2021-04,52,52,None,<NA>,0,unobserved_early_source,4,4,NaN


**Tên cột và kiểu dữ liệu**

,Cột,Kiểu Arrow,Kiểu pandas
0,client_id,large_string,str
1,month,large_string,str
2,trx_events,int64,Int64
3,trx_raw_events,uint32,uint32
4,trx_missing_reason,null,object
5,geo_events,int64,Int64
6,geo_raw_events,uint32,uint32
7,geo_missing_reason,large_string,str
8,dialog_events,int64,Int64
9,dialog_raw_events,uint32,uint32


## coverage_selection


In [16]:
coverage_selection = preview_table("coverage_selection")


**5 dòng mẫu · 3 cột**

,group,display_id,client_id
0,Nhiều nhất,KH-1,9162f460905b0c6838d781fb85f47616ae17e06ea3af99c50fd2b3153f2afe33
1,Nhiều nhất,KH-2,0e57a93a7030cf99bdba679b479d0b0581e6ef17bc55d72cd7bdfe9440ecade1
2,Nhiều nhất,KH-3,89edf6431f1d2782d3f9e2cd6313d63fb6bec7385fcfd166ce62aef7e45c0f25
3,Nhiều nhất,KH-4,7ba553f1fb0b17639e38a6e5e13a2088b0d7a00eff940fb6d9b46737ddd89f4d
4,Nhiều nhất,KH-5,476eed0bb7c8f20ec246600364d03ea7db12614b4f5781a6e51da45d3f6195ff


**Tên cột và kiểu dữ liệu**

,Cột,Kiểu Arrow,Kiểu pandas
0,group,large_string,str
1,display_id,large_string,str
2,client_id,large_string,str
